# Personalized news recommendation POC

This notebook compares topic-aware keyword retrieval, BM25-semantic hybrid retrieval, and BM25-semantic hybrid retrieval with Cohere reranking. Every personal and professional interest topic independently contributes its top five articles. It intentionally keeps **article relevance** (what to send) separate from **outreach priority** (whom to contact now).

Data minimization: only professional/personal interests, role, account context, and the few structured fields needed for the separate outreach score are loaded. Email, mobile, LinkedIn, communication channel, owners/guardians, helper columns, and navigation sheets are excluded.

API references: [Tavily Search](https://docs.tavily.com/documentation/api-reference/endpoint/search), [OpenAI embeddings](https://platform.openai.com/docs/guides/embeddings), and [Cohere Rerank](https://docs.cohere.com/v2/docs/rerank).

In [ ]:
%pip install -q pandas numpy scikit-learn openpyxl openai tavily-python cohere python-dotenv truststore

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time
from collections import Counter
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

import cohere
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity
from tavily import TavilyClient
from tavily.errors import TimeoutError as TavilyTimeoutError

load_dotenv()

from embedding import EmbeddingClient, get_embedding_model

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 30)

INPUT_XLSX = Path('CRM_Input_template.xlsx')
CACHE_DIR = Path('poc_cache')
CACHE_DIR.mkdir(exist_ok=True)

LOOKBACK_DAYS = 30
TAVILY_MAX_RESULTS = 10
TAVILY_SEARCH_DEPTH = 'advanced'
TAVILY_TIMEOUT_SECONDS = 90
TAVILY_MAX_RETRIES = 4
TAVILY_RETRY_BASE_SECONDS = 5.0
EMBEDDING_MODEL = get_embedding_model()
RERANK_MODEL = 'rerank-v4.0-pro'
RERANK_CANDIDATES = 20
FINAL_RERANK_WEIGHT = 0.65
TOP_RESULTS_PER_TOPIC = 5
BM25_K1 = 1.5
BM25_B = 0.75
BM25_SEMANTIC_WEIGHT = 0.60
BM25_LEXICAL_WEIGHT = 0.40
REFRESH_NEWS = False

assert math.isclose(BM25_SEMANTIC_WEIGHT + BM25_LEXICAL_WEIGHT, 1.0)

## 1. Inspect the workbook and prepare a minimal CRM dataset

The two source sheets used below are `Input Contact Information` and `Accounts Overview`. `Methodology` is read only to make the tier/cadence assumptions auditable. Calculated overview sheets are deliberately not imported.

In [ ]:
def inspect_workbook(path: Path) -> pd.DataFrame:
    book = pd.ExcelFile(path, engine='openpyxl')
    rows = []
    for sheet in book.sheet_names:
        preview = pd.read_excel(path, sheet_name=sheet, nrows=0, engine='openpyxl')
        rows.append({'sheet': sheet, 'columns': list(preview.columns)})
    return pd.DataFrame(rows)

workbook_inventory = inspect_workbook(INPUT_XLSX)
display(workbook_inventory)

contact_columns = [
    'Company Name', 'Contact Person Full Name', 'Designation', 'Seniority',
    'Relationship Quality', 'Automatic tier assigned',
    'Manual Tier change?', 'Professional interests Interests',
    'Personal Interests', 'Last engagement date',
]
account_columns = [
    'Account Name', 'Sector', 'Products/ Services', 'Competitors', 'Headquarters',
]

contacts_raw = pd.read_excel(
    INPUT_XLSX, sheet_name='Input Contact Information',
    usecols=contact_columns, engine='openpyxl',
)
accounts_raw = pd.read_excel(
    INPUT_XLSX, sheet_name='Accounts Overview',
    usecols=account_columns, engine='openpyxl',
)
methodology = pd.read_excel(
    INPUT_XLSX, sheet_name='Methodology', header=None, nrows=10, engine='openpyxl'
)
display(methodology.dropna(how='all'))

In [ ]:
def clean_text(value) -> str:
    if pd.isna(value):
        return ''
    return ' '.join(str(value).split())

contacts = contacts_raw.rename(columns={
    'Company Name': 'company',
    'Contact Person Full Name': 'contact',
    'Designation': 'designation',
    'Seniority': 'seniority',
    'Relationship Quality': 'relationship_quality',
    'Automatic tier assigned': 'automatic_tier',
    'Manual Tier change?': 'manual_tier',
    'Professional interests Interests': 'professional_interests',
    'Personal Interests': 'personal_interests',
    'Last engagement date': 'last_engagement_date',
})
accounts = accounts_raw.rename(columns={
    'Account Name': 'company', 'Sector': 'sector',
    'Products/ Services': 'products_services',
    'Competitors': 'competitors', 'Headquarters': 'headquarters',
})

contacts = contacts[contacts['contact'].notna() & contacts['company'].notna()].copy()
accounts = accounts[accounts['company'].notna()].copy()
contacts = contacts.drop_duplicates(['contact', 'company'], keep='first')
accounts = accounts.drop_duplicates('company', keep='first')

for col in contacts.columns.drop('last_engagement_date'):
    contacts[col] = contacts[col].map(clean_text)
for col in accounts.columns:
    accounts[col] = accounts[col].map(clean_text)
contacts['last_engagement_date'] = pd.to_datetime(
    contacts['last_engagement_date'], errors='coerce'
)
contacts['tier'] = contacts['manual_tier'].where(
    contacts['manual_tier'].ne(''), contacts['automatic_tier']
)

crm = contacts.merge(accounts, on='company', how='left', validate='many_to_one')
missing_accounts = crm.loc[crm['sector'].eq('') | crm['sector'].isna(), 'company'].unique()
assert len(missing_accounts) == 0, f'Missing account context for: {missing_accounts}'

crm['professional_profile'] = (
    'Role: ' + crm['designation'] + '. Professional interests: ' + crm['professional_interests']
).str.strip()
crm['personal_profile'] = crm['personal_interests'].str.strip()
crm['account_context'] = (
    'Company: ' + crm['company'] + '. Sector: ' + crm['sector'] +
    '. Products/services: ' + crm['products_services'] +
    '. Competitors: ' + crm['competitors'] +
    '. Geography: ' + crm['headquarters']
).str.strip()
crm['contact_id'] = [f'C{i:04d}' for i in range(1, len(crm) + 1)]

profile_columns = [
    'contact_id', 'contact', 'company', 'designation',
    'professional_interests', 'personal_interests',
    'professional_profile', 'personal_profile', 'account_context',
]
print(f'{len(crm)} contacts across {crm.company.nunique()} accounts')
display(crm[profile_columns].head())

### Separate outreach-priority score

This score is never used in article retrieval or ranking. It combines structured CRM urgency signals only. The cadence follows the workbook methodology: Tier 1 = 30 days, Tier 2 = 90, Tier 3 = 180.

In [ ]:
LEVEL_SCORE = {'High': 1.0, 'Medium': 0.6, 'Low': 0.2}
TIER_SCORE = {'Tier 1': 1.0, 'Tier 2': 0.6, 'Tier 3': 0.2}
CADENCE_DAYS = {'Tier 1': 30, 'Tier 2': 90, 'Tier 3': 180}
REVIEW_THRESHOLD_DAYS = {'Tier 1': 60, 'Tier 2': 180, 'Tier 3': 365}
as_of = pd.Timestamp.now(tz='UTC').tz_localize(None).normalize()

crm['days_since_engagement'] = (as_of - crm['last_engagement_date']).dt.days
crm['target_cadence_days'] = crm['tier'].map(CADENCE_DAYS).fillna(180)
crm['review_threshold_days'] = crm['tier'].map(REVIEW_THRESHOLD_DAYS).fillna(365)
crm['engagement_urgency'] = np.where(
    crm['days_since_engagement'].isna(),
    1.0,
    (crm['days_since_engagement'] / crm['target_cadence_days']).clip(0, 2) / 2,
)
crm['engagement_status'] = np.select(
    [
        crm['last_engagement_date'].isna(),
        crm['days_since_engagement'] <= crm['target_cadence_days'],
        crm['days_since_engagement'] <= crm['review_threshold_days'],
    ],
    ['No Task', 'On Track', 'For Review'],
    default='Overdue',
)
crm['outreach_priority_score'] = 100 * (
    0.25 * crm['seniority'].map(LEVEL_SCORE).fillna(0) +
    0.25 * crm['tier'].map(TIER_SCORE).fillna(0) +
    0.20 * crm['relationship_quality'].map(LEVEL_SCORE).fillna(0) +
    0.30 * crm['engagement_urgency']
)
outreach_priority = crm[[
    'contact', 'company', 'seniority', 'tier', 'relationship_quality',
    'last_engagement_date', 'days_since_engagement', 'target_cadence_days',
    'review_threshold_days',
    'engagement_status', 'outreach_priority_score',
]].sort_values('outreach_priority_score', ascending=False)
display(outreach_priority.head(10))

## 2. Retrieve and deduplicate approximately 30 days of news

Queries are deliberately batched: one account query and one aggregated professional-theme query per company, plus one query per distinct personal-interest theme. This improves coverage without issuing a separate search for every contact-interest pair. The cache filename includes the date window and query set.

In [ ]:
def interest_labels(text: str) -> list[str]:
    labels = []
    for part in clean_text(text).split(';'):
        label = re.split(r'\s+[—–-]\s+', part.strip(), maxsplit=1)[0].strip()
        if label:
            labels.append(label)
    return labels

def build_search_queries(crm_df: pd.DataFrame) -> pd.DataFrame:
    queries = []
    for company, group in crm_df.groupby('company', sort=True):
        row = group.iloc[0]
        competitors = '; '.join(clean_text(row['competitors']).split(';')[:3])
        queries.append({
            'discovery_type': 'account',
            'discovery_query': f'\"{company}\" {row.sector} {row.products_services} {competitors} latest news',
        })
        professional_themes = sorted({
            theme for text in group['professional_interests'] for theme in interest_labels(text)
        })
        for theme in professional_themes:
            queries.append({
                'discovery_type': 'professional',
                'discovery_query': (
                    f'{row.sector} {row.products_services} {theme} '
                    'latest news events developments'
                ),
            })
    personal_themes = sorted({
        theme for text in crm_df['personal_interests'] for theme in interest_labels(text)
    })
    for theme in personal_themes:
        queries.append({
            'discovery_type': 'personal',
            'discovery_query': f'{theme} latest news events developments',
        })
    return pd.DataFrame(queries).drop_duplicates('discovery_query').reset_index(drop=True)

search_queries = build_search_queries(crm)
display(search_queries.groupby('discovery_type').size().rename('query_count'))
display(search_queries.head())

In [ ]:
def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise EnvironmentError(f'Set {name} in the environment before running this stage.')
    return value

TRACKING_PARAMS = {'fbclid', 'gclid', 'mc_cid', 'mc_eid'}

def canonical_url(url: str) -> str:
    try:
        parsed = urlparse(url)
        query = [
            (k, v) for k, v in parse_qsl(parsed.query, keep_blank_values=True)
            if not k.lower().startswith('utm_') and k.lower() not in TRACKING_PARAMS
        ]
        return urlunparse((parsed.scheme.lower(), parsed.netloc.lower(), parsed.path.rstrip('/'), '', urlencode(query), ''))
    except Exception:
        return clean_text(url)

def news_cache_path(query_df: pd.DataFrame, start_date: str, end_date: str) -> Path:
    payload = query_df.to_json(orient='records') + start_date + end_date
    digest = hashlib.sha256(payload.encode()).hexdigest()[:12]
    return CACHE_DIR / f'articles_{start_date}_{end_date}_{digest}.json'

def write_json_atomic(path: Path, value) -> None:
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2))
    temporary.replace(path)

def retrieve_news(query_df: pd.DataFrame, refresh: bool = False) -> pd.DataFrame:
    end = pd.Timestamp.now(tz='UTC').date()
    start = end - pd.Timedelta(days=LOOKBACK_DAYS)
    start_s, end_s = start.isoformat(), end.isoformat()
    cache_path = news_cache_path(query_df, start_s, end_s)
    if cache_path.exists() and not refresh:
        return pd.DataFrame(json.loads(cache_path.read_text()))

    partial_path = cache_path.with_suffix('.partial.json')
    if partial_path.exists() and not refresh:
        progress = json.loads(partial_path.read_text())
        discovered = progress.get('discovered', [])
        completed_queries = set(progress.get('completed_queries', []))
        print(
            f'Resuming Tavily retrieval: {len(completed_queries)}/'
            f'{len(query_df)} searches already completed.'
        )
    else:
        discovered = []
        completed_queries = set()

    client = TavilyClient(api_key=require_env('TAVILY_API_KEY'))
    failed_queries = []
    for query_number, row in enumerate(query_df.itertuples(index=False), start=1):
        query = row.discovery_query
        if query in completed_queries:
            continue
        response = None
        for attempt in range(1, TAVILY_MAX_RETRIES + 1):
            try:
                response = client.search(
                    query=query, topic='news',
                    search_depth=(
                        TAVILY_SEARCH_DEPTH if attempt <= 2 else 'basic'
                    ),
                    max_results=TAVILY_MAX_RESULTS,
                    start_date=start_s, end_date=end_s,
                    include_published_date=True,
                    filter_by_published_date=True,
                    include_answer=False, include_raw_content=False,
                    timeout=TAVILY_TIMEOUT_SECONDS,
                )
                break
            except (TavilyTimeoutError, TimeoutError):
                if attempt == TAVILY_MAX_RETRIES:
                    failed_queries.append(query)
                    print(
                        f'Tavily timed out for search {query_number}/'
                        f'{len(query_df)} after {attempt} attempts; saved for retry.'
                    )
                    break
                wait_seconds = TAVILY_RETRY_BASE_SECONDS * (2 ** (attempt - 1))
                print(
                    f'Tavily timeout for search {query_number}/{len(query_df)}; '
                    f'retrying in {wait_seconds:.0f}s '
                    f'({attempt + 1}/{TAVILY_MAX_RETRIES}).'
                )
                time.sleep(wait_seconds)
        if response is None:
            continue
        for result in response.get('results', []):
            url = clean_text(result.get('url'))
            if not url:
                continue
            discovered.append({
                'title': clean_text(result.get('title')),
                'url': url,
                'canonical_url': canonical_url(url),
                'source': urlparse(url).netloc.lower().removeprefix('www.'),
                'publication_date': result.get('published_date'),
                'content': clean_text(result.get('content')),
                'discovery_query': query,
                'discovery_type': row.discovery_type,
            })
        completed_queries.add(query)
        write_json_atomic(partial_path, {
            'completed_queries': sorted(completed_queries),
            'discovered': discovered,
        })
        if len(completed_queries) % 10 == 0:
            print(f'Completed {len(completed_queries)}/{len(query_df)} searches')
        time.sleep(0.15)

    if failed_queries:
        print(
            f'Warning: {len(failed_queries)} Tavily searches remain unfinished. '
            'Continuing with checkpointed results; rerun this cell later to retry '
            'only those searches.'
        )

    raw = pd.DataFrame(discovered)
    if raw.empty:
        raise RuntimeError('Tavily returned no dated news results for the configured window.')
    raw['dedup_key'] = raw['canonical_url'].where(
        raw['canonical_url'].ne(''), raw['title'].str.lower().str.replace(r'\W+', ' ', regex=True).str.strip()
    )
    articles = (raw.groupby('dedup_key', as_index=False)
        .agg({
            'title': 'first', 'url': 'first', 'canonical_url': 'first',
            'source': 'first', 'publication_date': 'first', 'content': 'first',
            'discovery_query': lambda s: ' || '.join(sorted(set(s))),
            'discovery_type': lambda s: ' || '.join(sorted(set(s))),
        }))
    articles['publication_date'] = pd.to_datetime(articles['publication_date'], errors='coerce', utc=True)
    articles = articles[articles['title'].ne('') & articles['content'].ne('')].copy()
    articles['article_id'] = [f'A{i:05d}' for i in range(1, len(articles) + 1)]
    articles['article_text'] = 'Title: ' + articles['title'] + '. Summary: ' + articles['content']
    records = articles.assign(
        publication_date=articles['publication_date'].astype(str)
    ).to_dict(orient='records')
    if failed_queries:
        write_json_atomic(partial_path, {
            'completed_queries': sorted(completed_queries),
            'discovered': discovered,
        })
    else:
        write_json_atomic(cache_path, records)
        partial_path.unlink(missing_ok=True)
    return articles

articles = retrieve_news(search_queries, refresh=REFRESH_NEWS)
articles['publication_date'] = pd.to_datetime(articles['publication_date'], errors='coerce', utc=True)
if 'article_text' not in articles:
    articles['article_text'] = 'Title: ' + articles['title'] + '. Summary: ' + articles['content']
print(f'{len(articles)} deduplicated articles')
display(articles[['title', 'source', 'publication_date', 'discovery_type']].head())

## 3. Generate and cache embeddings

Only the individual interest-topic labels and article text are embedded. Operational CRM fields never enter an embedding request. The local cache is keyed by model and exact text.

In [ ]:
import importlib
from dotenv import load_dotenv

load_dotenv(".env", override=True)

import portkey_profiles
import embedding

importlib.reload(portkey_profiles)
embedding = importlib.reload(embedding)

EmbeddingClient = embedding.EmbeddingClient
EMBEDDING_MODEL = embedding.get_embedding_model()

print("Embedding module:", embedding.__file__)
print("Embedding model:", EMBEDDING_MODEL)

In [ ]:
EMBED_CACHE_PATH = CACHE_DIR / 'embeddings.json'

def load_json_dict(path: Path) -> dict:
    return json.loads(path.read_text()) if path.exists() else {}

def save_json_dict(path: Path, value: dict) -> None:
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value))
    temporary.replace(path)

def embedding_key(text: str) -> str:
    return hashlib.sha256(f'{EMBEDDING_MODEL}\n{text}'.encode()).hexdigest()

async def embed_texts(texts: list[str], batch_size: int = 100) -> np.ndarray:
    texts = [clean_text(text) for text in texts]
    cache = load_json_dict(EMBED_CACHE_PATH)
    missing = list(dict.fromkeys(text for text in texts if embedding_key(text) not in cache))
    if missing:
        client = EmbeddingClient(model=EMBEDDING_MODEL, batch_size=batch_size)
        vectors = await client.embed_texts_async(missing, normalize=False)
        for text, vector in zip(missing, vectors):
            cache[embedding_key(text)] = vector.tolist()
        save_json_dict(EMBED_CACHE_PATH, cache)
    return np.asarray([cache[embedding_key(text)] for text in texts], dtype=np.float32)

all_interest_topics = sorted({
    topic
    for column in ('professional_interests', 'personal_interests')
    for value in crm[column]
    for topic in interest_labels(value)
})
article_embeddings = await embed_texts(articles['article_text'].tolist())
topic_embeddings = await embed_texts(all_interest_topics)
topic_embedding_lookup = dict(zip(all_interest_topics, topic_embeddings))
print(
    'Embedding matrices:', article_embeddings.shape, topic_embeddings.shape
)

## 4. Keyword and BM25–semantic hybrid retrieval

Each professional and personal interest topic is scored independently. The BM25 hybrid is a transparent weighted mix of normalized BM25 lexical relevance and topic-level semantic similarity.

In [ ]:
STOPWORDS = {
    'the', 'and', 'for', 'with', 'that', 'this', 'from', 'into', 'their', 'about',
    'interested', 'interest', 'interests', 'follows', 'enjoys', 'latest', 'news',
    'role', 'company', 'sector', 'products', 'services', 'geography', 'major',
}

def token_list(text: str) -> list[str]:
    return [
        token for token in re.findall(r'(?u)\b[\w-]{3,}\b', clean_text(text).lower())
        if token not in STOPWORDS and not token.isdigit()
    ]

def tokens(text: str) -> set[str]:
    return set(token_list(text))

def keyword_relevance(query_text: str, article_text: str) -> float:
    query_tokens, article_tokens = tokens(query_text), tokens(article_text)
    if not query_tokens:
        return 0.0
    return min(1.0, len(query_tokens & article_tokens) / max(1.0, math.sqrt(len(query_tokens))))

ARTICLE_TOKEN_LISTS = [token_list(text) for text in articles['article_text']]
ARTICLE_TERM_FREQUENCIES = [Counter(items) for items in ARTICLE_TOKEN_LISTS]
ARTICLE_LENGTHS = np.asarray([len(items) for items in ARTICLE_TOKEN_LISTS], dtype=float)
BM25_AVERAGE_DOCUMENT_LENGTH = float(ARTICLE_LENGTHS.mean()) if len(ARTICLE_LENGTHS) else 0.0
BM25_DOCUMENT_FREQUENCY = Counter()
for items in ARTICLE_TOKEN_LISTS:
    BM25_DOCUMENT_FREQUENCY.update(set(items))
BM25_IDF = {
    term: math.log(1 + (len(ARTICLE_TOKEN_LISTS) - frequency + 0.5) / (frequency + 0.5))
    for term, frequency in BM25_DOCUMENT_FREQUENCY.items()
}

def bm25_relevance(query_text: str) -> np.ndarray:
    query_terms = token_list(query_text)
    scores = np.zeros(len(ARTICLE_TOKEN_LISTS), dtype=float)
    if not query_terms or BM25_AVERAGE_DOCUMENT_LENGTH == 0:
        return scores
    for term in query_terms:
        idf = BM25_IDF.get(term, 0.0)
        if idf == 0:
            continue
        for index, frequencies in enumerate(ARTICLE_TERM_FREQUENCIES):
            frequency = frequencies.get(term, 0)
            if frequency == 0:
                continue
            length_factor = 1 - BM25_B + BM25_B * (ARTICLE_LENGTHS[index] / BM25_AVERAGE_DOCUMENT_LENGTH)
            scores[index] += idf * (frequency * (BM25_K1 + 1)) / (frequency + BM25_K1 * length_factor)
    return scores

def normalize_nonnegative(values: np.ndarray) -> np.ndarray:
    maximum = float(np.max(values)) if len(values) else 0.0
    return values / maximum if maximum > 0 else np.zeros_like(values, dtype=float)

def score_all_candidates() -> pd.DataFrame:
    frames = []
    article_records = articles.reset_index(drop=True)
    for contact_idx, contact_row in crm.reset_index(drop=True).iterrows():
        for category in ('professional', 'personal'):
            topics = interest_labels(contact_row[f'{category}_interests'])
            for topic in topics:
                topic_vector = topic_embedding_lookup[topic]
                semantic_scores = cosine_similarity(
                    np.asarray([topic_vector]), article_embeddings
                )[0]
                keyword_scores = np.asarray([
                    keyword_relevance(topic, text) for text in article_records['article_text']
                ])
                bm25_scores = normalize_nonnegative(bm25_relevance(topic))
                bm25_hybrid_scores = (
                    BM25_SEMANTIC_WEIGHT * np.clip(semantic_scores, 0, 1) +
                    BM25_LEXICAL_WEIGHT * bm25_scores
                )
                frame = article_records[[
                    'article_id', 'title', 'url', 'source', 'publication_date',
                    'content', 'article_text', 'discovery_query', 'discovery_type',
                ]].copy()
                frame.insert(0, 'company', contact_row.company)
                frame.insert(0, 'contact', contact_row.contact)
                frame.insert(0, 'contact_id', contact_row.contact_id)
                frame['category'] = category
                frame['interest_topic'] = topic
                frame['keyword_score'] = keyword_scores
                frame['semantic_similarity'] = semantic_scores
                frame['bm25_score'] = bm25_scores
                frame['bm25_hybrid_score'] = bm25_hybrid_scores
                frames.append(frame)
    return pd.concat(frames, ignore_index=True)

candidate_scores = score_all_candidates()
print(f'{len(candidate_scores):,} contact/category/topic/article score rows')

## 5. Export non-reranked BM25 results, then rerank with Cohere

The evaluation workbook is written before the first Cohere request, so it remains independent of reranking and is available even if the trial-key reranker is interrupted. Reranking is then performed independently for every contact/category/topic group, preserving topic coverage.

In [ ]:
TOPIC_GROUP_COLUMNS = ['contact_id', 'category', 'interest_topic']

def top_results_per_topic(
    frame: pd.DataFrame, score_column: str, method_name: str
) -> pd.DataFrame:
    selected_groups = []
    for _, group in frame.groupby(TOPIC_GROUP_COLUMNS, sort=False):
        selected = group.nlargest(TOP_RESULTS_PER_TOPIC, score_column).copy()
        selected['rank'] = range(1, len(selected) + 1)
        selected['retrieval_method'] = method_name
        selected_groups.append(selected)
    return pd.concat(selected_groups, ignore_index=True)

def assert_topic_coverage(frame: pd.DataFrame, method_name: str) -> None:
    counts = frame.groupby(TOPIC_GROUP_COLUMNS).size()
    if not counts.eq(TOP_RESULTS_PER_TOPIC).all():
        incomplete = counts[counts.ne(TOP_RESULTS_PER_TOPIC)]
        raise AssertionError(
            f'{method_name} does not contain exactly {TOP_RESULTS_PER_TOPIC} rows '
            f'for every topic: {incomplete.to_dict()}'
        )

keyword_results = top_results_per_topic(
    candidate_scores, 'keyword_score', 'Keyword'
)
bm25_hybrid_results = top_results_per_topic(
    candidate_scores, 'bm25_hybrid_score', 'BM25 Hybrid'
)
assert_topic_coverage(keyword_results, 'Keyword')
assert_topic_coverage(bm25_hybrid_results, 'BM25 Hybrid')

COMPARISON_COLUMNS = [
    'contact', 'company', 'category', 'interest_topic', 'article_title',
    'source', 'publication_date', 'url', 'retrieval_method', 'rank',
    'keyword_score', 'bm25_score', 'semantic_similarity',
    'bm25_hybrid_score', 'reranker_score', 'bm25_reranked_score',
]

def comparison_export(frame: pd.DataFrame) -> pd.DataFrame:
    exported = frame.rename(columns={'title': 'article_title'}).copy()
    for column in COMPARISON_COLUMNS:
        if column not in exported:
            exported[column] = np.nan
    exported['publication_date'] = exported['publication_date'].dt.tz_localize(None)
    return exported[COMPARISON_COLUMNS].sort_values(
        ['contact', 'category', 'interest_topic', 'rank']
    )

keyword_export = comparison_export(keyword_results)
bm25_hybrid_export = comparison_export(bm25_hybrid_results)
bm25_evaluation = bm25_hybrid_export[[
    'contact', 'company', 'category', 'interest_topic', 'article_title',
    'source', 'publication_date', 'url', 'rank',
]].copy()
bm25_evaluation['human_relevance_score'] = pd.Series(
    pd.NA, index=bm25_evaluation.index, dtype='Int64'
)
bm25_evaluation['human_topic_match'] = ''
bm25_evaluation['human_outreach_appropriate'] = ''
bm25_evaluation['human_comments'] = ''
bm25_evaluation.to_excel(
    'personalized_news_bm25_hybrid_evaluation.xlsx',
    sheet_name='BM25 Hybrid Evaluation', index=False, engine='openpyxl'
)
print(
    f'Exported {len(bm25_evaluation):,} non-reranked BM25 hybrid rows '
    'for team evaluation.'
)

RERANK_CACHE_PATH = CACHE_DIR / 'reranks.json'
RERANK_MIN_INTERVAL_SECONDS = 6.5  # Trial keys allow 10 rerank requests/minute.
RERANK_MAX_RATE_LIMIT_RETRIES = 4

def rerank_query(contact_row: pd.Series, category: str, topic: str) -> str:
    if category == 'professional':
        focus = (
            f'Professional topic: {topic}. Role: {contact_row.designation}. '
            f'Account context: {contact_row.account_context}'
        )
    else:
        focus = (
            f'Personal topic only: {topic}. Do not use professional interests or company context.'
        )
    return (
        'Rank how genuinely relevant each article is to the exact requested topic. '
        f'{focus} Do not rank an article highly merely because it shares isolated keywords.'
    )

def rerank_top_candidates(scores: pd.DataFrame) -> pd.DataFrame:
    client = cohere.ClientV2(api_key=require_env('COHERE_API_KEY'))
    cache = load_json_dict(RERANK_CACHE_PATH)
    outputs = []
    crm_lookup = crm.set_index('contact_id')
    grouped = scores.groupby(['contact_id', 'category', 'interest_topic'], sort=False)
    last_request_at = 0.0
    for group_number, ((contact_id, category, interest_topic), group) in enumerate(grouped, start=1):
        candidates = group.nlargest(
            RERANK_CANDIDATES, 'bm25_hybrid_score'
        ).copy().reset_index(drop=True)
        contact_row = crm_lookup.loc[contact_id]
        query = rerank_query(contact_row, category, interest_topic)
        documents = [
            f'Title: {row.title}\nSource: {row.source}\nPublished: {row.publication_date}\nSummary: {row.content[:2500]}'
            for row in candidates.itertuples()
        ]
        cache_key = hashlib.sha256(
            (RERANK_MODEL + '\n' + query + '\n' + '\n---\n'.join(documents)).encode()
        ).hexdigest()
        if cache_key not in cache:
            elapsed = time.monotonic() - last_request_at
            if elapsed < RERANK_MIN_INTERVAL_SECONDS:
                time.sleep(RERANK_MIN_INTERVAL_SECONDS - elapsed)
            for attempt in range(1, RERANK_MAX_RATE_LIMIT_RETRIES + 1):
                try:
                    last_request_at = time.monotonic()
                    response = client.rerank(
                        model=RERANK_MODEL, query=query, documents=documents, top_n=len(documents)
                    )
                    break
                except cohere.TooManyRequestsError as exc:
                    if attempt == RERANK_MAX_RATE_LIMIT_RETRIES:
                        raise
                    retry_after = (getattr(exc, 'headers', {}) or {}).get('retry-after')
                    try:
                        wait_seconds = max(RERANK_MIN_INTERVAL_SECONDS, float(retry_after))
                    except (TypeError, ValueError):
                        wait_seconds = 65.0
                    print(
                        f'Cohere trial limit reached; waiting {wait_seconds:.0f}s '
                        f'before retry {attempt + 1}/{RERANK_MAX_RATE_LIMIT_RETRIES}.'
                    )
                    time.sleep(wait_seconds)
            cache[cache_key] = [
                {'index': int(item.index), 'relevance_score': float(item.relevance_score)}
                for item in response.results
            ]
            save_json_dict(RERANK_CACHE_PATH, cache)
        score_by_index = {item['index']: item['relevance_score'] for item in cache[cache_key]}
        candidates['reranker_score'] = [score_by_index.get(i, 0.0) for i in range(len(candidates))]
        candidates['bm25_reranked_score'] = (
            FINAL_RERANK_WEIGHT * candidates['reranker_score'] +
            (1 - FINAL_RERANK_WEIGHT) * candidates['bm25_hybrid_score']
        )
        outputs.append(candidates)
        if group_number % 10 == 0:
            print(f'Reranked {group_number}/{len(grouped)} contact/category groups')
    return pd.concat(outputs, ignore_index=True)

reranked_candidates = rerank_top_candidates(candidate_scores)
print(f'{len(reranked_candidates):,} reranked candidate rows')

## 6. Export the three-method comparison workbook

In [ ]:
bm25_reranked_results = top_results_per_topic(
    reranked_candidates, 'bm25_reranked_score', 'BM25 Hybrid + Reranker'
)
assert_topic_coverage(bm25_reranked_results, 'BM25 Hybrid + Reranker')
bm25_reranked_export = comparison_export(bm25_reranked_results)

with pd.ExcelWriter(
    'personalized_news_retrieval_comparison.xlsx', engine='openpyxl'
) as writer:
    keyword_export.to_excel(writer, sheet_name='Keyword', index=False)
    bm25_hybrid_export.to_excel(writer, sheet_name='BM25 Hybrid', index=False)
    bm25_reranked_export.to_excel(
        writer, sheet_name='BM25 Hybrid Reranked', index=False
    )
print(
    f'Exported {len(keyword_export):,} rows per method to '
    'personalized_news_retrieval_comparison.xlsx.'
)
display(bm25_hybrid_export.head(10))

## 7. Inspect one contact across methods

Each table shows five results for every personal and professional interest topic. The reranked method reranks the top BM25-hybrid candidates inside each topic, so one topic cannot crowd out another.

In [ ]:
DISPLAY_COLUMNS = [
    'category', 'interest_topic', 'rank', 'title', 'source', 'publication_date',
    'keyword_score', 'bm25_score', 'semantic_similarity', 'bm25_hybrid_score',
]

def contact_method_results(contact_name: str, frame: pd.DataFrame) -> pd.DataFrame:
    data = frame[
        frame['contact'].str.casefold() == clean_text(contact_name).casefold()
    ].copy()
    optional = [
        column for column in ('reranker_score', 'bm25_reranked_score')
        if column in data.columns
    ]
    return data[DISPLAY_COLUMNS + optional].sort_values(
        ['category', 'interest_topic', 'rank']
    )

def show_contact_results(contact_name: str) -> None:
    matches = crm[crm['contact'].str.casefold() == clean_text(contact_name).casefold()]
    if matches.empty:
        suggestions = crm.loc[crm['contact'].str.contains(contact_name, case=False, na=False), 'contact'].tolist()
        raise KeyError(f'Contact not found. Partial matches: {suggestions}')
    contact_row = matches.iloc[0]
    display(pd.DataFrame([{
        'contact': contact_row.contact, 'company': contact_row.company,
        'designation': contact_row.designation,
        'professional_interests': contact_row.professional_interests,
        'personal_interests': contact_row.personal_interests,
        'account_context': contact_row.account_context,
        'outreach_priority_score (separate)': contact_row.outreach_priority_score,
    }]))
    for label, frame in (
        ('Keyword: top 5 per topic', keyword_results),
        ('BM25 hybrid without reranker: top 5 per topic', bm25_hybrid_results),
        ('BM25 hybrid with reranker: top 5 per topic', bm25_reranked_results),
    ):
        print(label)
        display(contact_method_results(contact_row.contact, frame))

show_contact_results(crm.iloc[0]['contact'])

## 8. Evaluate the completed BM25-hybrid review workbook

Give the separate `personalized_news_bm25_hybrid_evaluation.xlsx` workbook to reviewers. It contains only BM25-semantic hybrid results without reranking or model-score columns. Rating scale: **0** irrelevant, **1** weakly related, **2** relevant, **3** excellent. Precision@5 treats ratings 2–3 as relevant.

In [ ]:
# The evaluation workbook is created in section 5 with blank human-review fields.
# Reviewers should fill it in without consulting the comparison workbook.
display(bm25_evaluation.head(10))

In [ ]:
def precision_at_k(group: pd.DataFrame, k: int = 5, relevant_threshold: int = 2) -> float:
    rated = group.sort_values('rank').head(k)['human_relevance_score'].dropna()
    return float((rated >= relevant_threshold).mean()) if len(rated) == k else np.nan

def evaluate_human_scores(
    path: str | Path = 'personalized_news_bm25_hybrid_evaluation.xlsx',
) -> pd.DataFrame:
    ratings = pd.read_excel(path, sheet_name='BM25 Hybrid Evaluation', engine='openpyxl')
    ratings['human_relevance_score'] = pd.to_numeric(ratings['human_relevance_score'], errors='coerce')
    invalid = ratings['human_relevance_score'].dropna()
    if not invalid.between(0, 3).all():
        raise ValueError('human_relevance_score must be blank or an integer from 0 through 3.')
    per_topic = (ratings.groupby(['contact', 'category', 'interest_topic'], dropna=False)
        .apply(precision_at_k, include_groups=False)
        .rename('precision_at_5').reset_index())
    rated = ratings.dropna(subset=['human_relevance_score'])
    summary = ratings.groupby('category', dropna=False)['human_relevance_score'].agg(
        average_human_relevance='mean', rated_articles='count'
    ).reset_index()
    percentages = rated.groupby('category', dropna=False)['human_relevance_score'].agg(
        percentage_relevant=lambda values: 100 * values.ge(2).mean(),
        percentage_scored_3=lambda values: 100 * values.eq(3).mean(),
        percentage_scored_0=lambda values: 100 * values.eq(0).mean(),
    ).reset_index()
    precision = per_topic.groupby('category', as_index=False)['precision_at_5'].mean()
    return summary.merge(precision, on='category', how='left').merge(
        percentages, on='category', how='left'
    )

# After reviewers fill the exported workbook, run:
# display(evaluate_human_scores())

## Reading the experiment

Use `show_contact_results(name)` to inspect keyword, BM25 hybrid, and reranked BM25 hybrid results topic by topic. Share only `personalized_news_bm25_hybrid_evaluation.xlsx` with reviewers; it intentionally omits reranker output and model scores. After reviewers fill the human fields, run `evaluate_human_scores()` to summarize Precision@5 and relevance by personal/professional category.